# Data Overview

This notebook inspects the DACON product sales dataset and converts the wide sales matrix into a long time-series table.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
train_path = DATA_DIR / "train.csv"
sample_submission_path = DATA_DIR / "sample_submission.csv"

In [ ]:
train = pd.read_csv(train_path)
sample_submission = pd.read_csv(sample_submission_path)

print(f"train: {train.shape}")
print(f"sample_submission: {sample_submission.shape}")
train.head()

In [ ]:
parsed_columns = pd.to_datetime(pd.Index(train.columns), errors="coerce")
date_columns = [column for column, parsed in zip(train.columns, parsed_columns) if not pd.isna(parsed)]
metadata_columns = [column for column in train.columns if column not in date_columns]

print(f"metadata columns: {metadata_columns}")
print(f"sales date range: {date_columns[0]} ~ {date_columns[-1]}")
print(f"sales days: {len(date_columns)}")

In [ ]:
sales_long = train.melt(
    id_vars=metadata_columns,
    value_vars=date_columns,
    var_name="date",
    value_name="sales",
)
sales_long["date"] = pd.to_datetime(sales_long["date"])

sales_long.head()

In [ ]:
daily_sales = sales_long.groupby("date", as_index=False)["sales"].sum()
daily_sales.describe()